# Academic Survival Longitudinal Data Analysis

Exploratory analysis of `academic_survival_longitudinal (1).csv` using pandas and matplotlib.

Dataset: longitudinal student records with 22 columns (Student_ID through Censored), multiple semesters per student.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)


In [ ]:
# Load the CSV file
df = pd.read_csv('academic_survival_longitudinal (1).csv')

# Preview the first rows
df.head()


In [ ]:
# Dataset shape and column info
print('Shape:', df.shape)
print()
df.info()

# Summary statistics
df.describe(include='all')


In [ ]:
# Check for missing values per column
df.isnull().sum()


In [ ]:
# Map Gender to four standard categories: Female, Male, Others, Prefer not to say
df['Gender'] = df['Gender'].astype(str).str.strip().str.title()
gender_map = {
    'Female': 'Female', 'Male': 'Male', 'M': 'Male', 'F': 'Female',
    'Other': 'Others', 'Others': 'Others',
    'Prefer Not To Say': 'Prefer not to say',
}
df['Gender'] = df['Gender'].replace(gender_map)
df['Gender'] = df['Gender'].where(
    df['Gender'].isin(['Female', 'Male', 'Others', 'Prefer not to say']), 'Others'
)
df['Gender'].value_counts()


In [ ]:
# Handle negative Sem_GPA: flag invalid values (< 0), replace with NaN
df['sem_gpa_invalid'] = df['Sem_GPA'] < 0
print('Invalid (negative) Sem_GPA rows:')
print(df.loc[df['Sem_GPA'] < 0, ['Student_ID', 'Semester', 'Sem_GPA']])

df.loc[df['Sem_GPA'] < 0, 'Sem_GPA'] = np.nan
df['Sem_GPA'] = df['Sem_GPA'].clip(lower=0, upper=4)

df['Sem_GPA'].describe()


In [ ]:
# Impute missing Family_Income with median per student (longitudinal), fallback to overall median
df['Family_Income'] = df.groupby('Student_ID')['Family_Income'].transform(
    lambda s: s.fillna(s.median())
)
df['Family_Income'] = df['Family_Income'].fillna(df['Family_Income'].median())

print('Remaining missing Family_Income:', df['Family_Income'].isnull().sum())
df['Family_Income'].describe()


In [ ]:
# Handle missing Attendance (STU_00022 semester 2): impute with student median, fallback overall median
df['Attendance'] = df.groupby('Student_ID')['Attendance'].transform(
    lambda s: s.fillna(s.median())
)
df['Attendance'] = df['Attendance'].fillna(df['Attendance'].median())

print('Remaining missing Attendance:', df['Attendance'].isnull().sum())
df['Attendance'].describe()


In [ ]:
# Emergency_Expense outliers > 1,000: cap at the 99th percentile
outliers = df.loc[df['Emergency_Expense'] > 1000, ['Student_ID', 'Semester', 'Emergency_Expense']]
print('Emergency_Expense outliers > 1,000:')
print(outliers)

cap = df['Emergency_Expense'].quantile(0.99)
df['Emergency_Expense'] = df['Emergency_Expense'].clip(upper=cap)

print('Cap value (99th percentile):', cap)
df['Emergency_Expense'].describe()


In [ ]:
# Drop exact duplicate rows
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'Dropped {before - after} duplicate rows')

df.head()


In [ ]:
# End_of_Semester_Status categories: Enrolled, Dropped_Out, Graduated
print('End_of_Semester_Status value counts:')
print(df['End_of_Semester_Status'].value_counts())
print()
print('Censored value counts (binary 0/1):')
print(df['Censored'].value_counts())
print()
print('Target_Dropout_Next_Sem value counts:')
print(df['Target_Dropout_Next_Sem'].value_counts())


In [ ]:
# Semester distribution per student
print('Semesters per student:')
print(df.groupby('Student_ID')['Semester'].nunique().describe())
print()
print('Sample student trajectories:')
print(df[df['Student_ID'].isin(['STU_00001', 'STU_00018'])][['Student_ID', 'Semester', 'Sem_GPA', 'End_of_Semester_Status', 'Censored', 'Target_Dropout_Next_Sem']])


In [ ]:
# Distribution of Sem_GPA
df['Sem_GPA'].hist(bins=30)
plt.title('Distribution of Sem_GPA')
plt.xlabel('Sem_GPA')
plt.ylabel('Count')
plt.show()

# Scatter plot: Emergency_Expense vs Sem_GPA
df.plot.scatter(x='Emergency_Expense', y='Sem_GPA', alpha=0.5)
plt.title('Emergency_Expense vs Sem_GPA')
plt.show()

# End_of_Semester_Status by Gender
pd.crosstab(df['Gender'], df['End_of_Semester_Status'])


In [ ]:
# EDA Table 1: Gender Ã— End_of_Semester_Status crosstab with row percentages
t1 = pd.crosstab(df['Gender'], df['End_of_Semester_Status'], margins=True)
t1_pct = t1.div(t1['All'], axis=0).mul(100).round(1)
print('Table 1: Gender Ã— End_of_Semester_Status (% of row)')
print(t1_pct)
print()

# EDA Table 2: Semester Ã— End_of_Semester_Status crosstab
t2 = pd.crosstab(df['Semester'], df['End_of_Semester_Status'], margins=True)
print('Table 2: Semester Ã— End_of_Semester_Status (counts)')
print(t2)
print()

# EDA Table 3: Scholarship Ã— End_of_Semester_Status crosstab with row percentages
t3 = pd.crosstab(df['Scholarship'], df['End_of_Semester_Status'], margins=True)
t3_pct = t3.div(t3['All'], axis=0).mul(100).round(1)
print('Table 3: Scholarship Ã— End_of_Semester_Status (% of row)')
print(t3_pct)
print()

# EDA Table 4: Family_Income binned stats by End_of_Semester_Status
df['Income_Bin'] = pd.qcut(df['Family_Income'], q=4, duplicates='drop')
t4 = df.groupby(['Income_Bin', 'End_of_Semester_Status']).size().unstack(fill_value=0)
t4['Total'] = t4.sum(axis=1)
t4_pct = t4.div(t4['Total'], axis=0).mul(100).round(1)
print('Table 4: Income quartile Ã— End_of_Semester_Status (% of row)')
print(t4_pct)
print()

# EDA Table 5: Summary metrics by Gender
t5 = df.groupby('Gender').agg(
    n_students=('Student_ID', 'nunique'),
    n_rows=('Student_ID', 'size'),
    avg_sem_gpa=('Sem_GPA', 'mean'),
    avg_attendance=('Attendance', 'mean'),
    avg_financial_stress=('Financial_Stress', 'mean'),
    dropout_rate=('Target_Dropout_Next_Sem', 'mean'),
).round(3)
t5['dropout_rate'] = (t5['dropout_rate'] * 100).round(1)
print('Table 5: Summary metrics by Gender')
print(t5)


In [ ]:
# Distribution Tables (counts & percentages)
dist_status = df['End_of_Semester_Status'].value_counts()
dist_status_pct = (dist_status / len(df) * 100).round(1)
dist_status_df = pd.DataFrame({'Count': dist_status, 'Percent': dist_status_pct})
print('Distribution Table A: End_of_Semester_Status')
print(dist_status_df)
print()

dist_gender = df['Gender'].value_counts()
dist_gender_pct = (dist_gender / len(df) * 100).round(1)
dist_gender_df = pd.DataFrame({'Count': dist_gender, 'Percent': dist_gender_pct})
print('Distribution Table B: Gender')
print(dist_gender_df)
print()

dist_scholarship = df['Scholarship'].value_counts()
dist_scholarship_pct = (dist_scholarship / len(df) * 100).round(1)
dist_scholarship_df = pd.DataFrame({'Count': dist_scholarship, 'Percent': dist_scholarship_pct})
print('Distribution Table C: Scholarship')
print(dist_scholarship_df)
print()

dist_semester = df['Semester'].value_counts().sort_index()
dist_semester_pct = (dist_semester / len(df) * 100).round(1)
dist_semester_df = pd.DataFrame({'Count': dist_semester, 'Percent': dist_semester_pct})
print('Distribution Table D: Semester')
print(dist_semester_df)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution Tables', fontsize=16, weight='bold', y=1.02)

colors_status = {'Enrolled': '#059669', 'Dropped_Out': '#dc2626', 'Graduated': '#0891b2'}
ax = axes[0, 0]
bars = ax.bar(dist_status.index, dist_status.values, color=[colors_status.get(x, '#64748b') for x in dist_status.index])
ax.set_title('End_of_Semester_Status Distribution', weight='bold')
ax.set_ylabel('Count')
for i, (idx, val) in enumerate(dist_status.items()):
    ax.text(i, val/2, f'{dist_status_pct[idx]}%', ha='center', va='center', color='white', weight='bold', fontsize=10)
ax.set_ylim(0, max(dist_status.values) * 1.15)

colors_g = {'Female': '#2563eb', 'Male': '#7c3aed', 'Others': '#d97706', 'Prefer not to say': '#be185d'}
ax = axes[0, 1]
bars = ax.bar(dist_gender.index, dist_gender.values, color=[colors_g.get(x, '#64748b') for x in dist_gender.index])
ax.set_title('Gender Distribution', weight='bold')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=15)
for i, (idx, val) in enumerate(dist_gender.items()):
    ax.text(i, val/2, f'{dist_gender_pct[idx]}%', ha='center', va='center', color='white', weight='bold', fontsize=10)
ax.set_ylim(0, max(dist_gender.values) * 1.15)

ax = axes[1, 0]
labels_scholarship = ['No Scholarship (0)', 'Scholarship (1)']
bars = ax.bar(labels_scholarship, dist_scholarship.values, color=['#94a3b8', '#2563eb'])
ax.set_title('Scholarship Distribution', weight='bold')
ax.set_ylabel('Count')
for i, (idx, val) in enumerate(dist_scholarship.items()):
    ax.text(i, val/2, f'{dist_scholarship_pct[idx]}%', ha='center', va='center', color='white', weight='bold', fontsize=10)
ax.set_ylim(0, max(dist_scholarship.values) * 1.15)

ax = axes[1, 1]
ax.bar(dist_semester.index, dist_semester.values, color='#0891b2')
ax.set_title('Semester Distribution', weight='bold')
ax.set_xlabel('Semester')
ax.set_ylabel('Count')
ax.set_xticks(dist_semester.index)
for i, (idx, val) in enumerate(dist_semester.items()):
    ax.text(idx, val/2, f'{dist_semester_pct[idx]}%', ha='center', va='center', color='white', weight='bold', fontsize=9)
ax.set_ylim(0, max(dist_semester.values) * 1.15)

plt.tight_layout()
plt.savefig('eda_distribution_tables.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_distribution_tables.png')


In [ ]:
# Line Charts: trends over semesters
semester_stats = df.groupby('Semester').agg(
    dropout_rate=('Target_Dropout_Next_Sem', 'mean'),
    avg_gpa=('Sem_GPA', 'mean'),
    n=('Student_ID', 'size')
).reset_index()
semester_stats['dropout_rate_pct'] = semester_stats['dropout_rate'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(semester_stats['Semester'], semester_stats['dropout_rate_pct'], marker='o', linewidth=2.5, markersize=8, color='#dc2626')
ax.set_title('Dropout Rate by Semester', weight='bold')
ax.set_xlabel('Semester')
ax.set_ylabel('Dropout Rate (%)')
ax.set_xticks(semester_stats['Semester'])
ax.grid(True, alpha=0.3)
for _, row in semester_stats.iterrows():
    ax.annotate(f'{row["dropout_rate_pct"]:.1f}%', (row['Semester'], row['dropout_rate_pct']),
                textcoords="offset points", xytext=(0, 10), ha='center', fontsize=9, weight='bold', color='#dc2626')

ax = axes[1]
ax.plot(semester_stats['Semester'], semester_stats['avg_gpa'], marker='s', linewidth=2.5, markersize=8, color='#059669')
ax.set_title('Average Sem_GPA by Semester', weight='bold')
ax.set_xlabel('Semester')
ax.set_ylabel('Average Sem_GPA')
ax.set_xticks(semester_stats['Semester'])
ax.set_ylim(0, 4.2)
ax.grid(True, alpha=0.3)
for _, row in semester_stats.iterrows():
    ax.annotate(f'{row["avg_gpa"]:.2f}', (row['Semester'], row['avg_gpa']),
                textcoords="offset points", xytext=(0, 10), ha='center', fontsize=9, weight='bold', color='#059669')

plt.tight_layout()
plt.savefig('eda_line_charts_trends.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_line_charts_trends.png')

fig, ax = plt.subplots(figsize=(10, 6))
ax2 = ax.twinx()
line1 = ax.plot(semester_stats['Semester'], semester_stats['n'], marker='o', linewidth=2.5, markersize=8, color='#2563eb', label='Records')
line2 = ax2.plot(semester_stats['Semester'], semester_stats['dropout_rate_pct'], marker='s', linewidth=2.5, markersize=8, color='#dc2626', label='Dropout Rate (%)')
ax.set_title('Student Records and Dropout Rate by Semester', weight='bold')
ax.set_xlabel('Semester')
ax.set_ylabel('Number of Records', color='#2563eb')
ax2.set_ylabel('Dropout Rate (%)', color='#dc2626')
ax.set_xticks(semester_stats['Semester'])
ax.grid(True, alpha=0.3)
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='upper right')
plt.tight_layout()
plt.savefig('eda_line_chart_enrollment_dropout.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_line_chart_enrollment_dropout.png')


In [ ]:
# Bar Graphs: dropout rate by category
gender_dropout = df.groupby('Gender')['Target_Dropout_Next_Sem'].mean() * 100
gender_dropout = gender_dropout.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar1 = ['#dc2626' if v == gender_dropout.max() else '#2563eb' for v in gender_dropout.values]
bars = ax.bar(gender_dropout.index, gender_dropout.values, color=colors_bar1)
ax.set_title('Dropout Rate by Gender', weight='bold')
ax.set_ylabel('Dropout Rate (%)')
ax.set_ylim(0, max(gender_dropout.values) * 1.15)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}%', ha='center', va='bottom', fontsize=11, weight='bold')
plt.tight_layout()
plt.savefig('eda_bar_dropout_by_gender.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_bar_dropout_by_gender.png')

scholarship_dropout = df.groupby('Scholarship')['Target_Dropout_Next_Sem'].mean() * 100
scholarship_dropout.index = ['No Scholarship (0)', 'Scholarship (1)']

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(scholarship_dropout.index, scholarship_dropout.values, color=['#94a3b8', '#2563eb'])
ax.set_title('Dropout Rate by Scholarship', weight='bold')
ax.set_ylabel('Dropout Rate (%)')
ax.set_ylim(0, max(scholarship_dropout.values) * 1.15)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}%', ha='center', va='bottom', fontsize=11, weight='bold')
plt.tight_layout()
plt.savefig('eda_bar_dropout_by_scholarship.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_bar_dropout_by_scholarship.png')

df['Income_Bin'] = pd.qcut(df['Family_Income'], q=4, duplicates='drop')
income_dropout = df.groupby('Income_Bin')['Target_Dropout_Next_Sem'].mean() * 100

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(range(len(income_dropout)), income_dropout.values, color=['#dc2626', '#ea580c', '#059669', '#0891b2'])
ax.set_title('Dropout Rate by Income Quartile', weight='bold')
ax.set_ylabel('Dropout Rate (%)')
ax.set_xlabel('Income Quartile (Q1=Lowest â†’ Q4=Highest)')
ax.set_xticks(range(len(income_dropout)))
ax.set_xticklabels(['Q1\n(0-35k)', 'Q2\n(35-52k)', 'Q3\n(52-77k)', 'Q4\n(77k+)'], fontsize=10)
ax.set_ylim(0, max(income_dropout.values) * 1.15)
for i, bar in enumerate(bars):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}%', ha='center', va='bottom', fontsize=11, weight='bold')
plt.tight_layout()
plt.savefig('eda_bar_dropout_by_income.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_bar_dropout_by_income.png')

gender_stress = df.groupby('Gender')['Financial_Stress'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(gender_stress.index, gender_stress.values, color=['#2563eb', '#7c3aed', '#d97706', '#be185d'])
ax.set_title('Average Financial Stress by Gender', weight='bold')
ax.set_ylabel('Average Financial Stress (1-5 scale)')
ax.set_ylim(0, max(gender_stress.values) * 1.15)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.2f}', ha='center', va='bottom', fontsize=11, weight='bold')
plt.tight_layout()
plt.savefig('eda_bar_stress_by_gender.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_bar_stress_by_gender.png')


In [ ]:
# Save cleaned data
df.to_csv('academic_survival_longitudinal_cleaned.csv', index=False)
print('Saved cleaned data to academic_survival_longitudinal_cleaned.csv')
print('Final shape:', df.shape)


## Next steps

- Confirm `academic_survival_longitudinal (1).csv` is in this folder before running.
- Install dependencies: `conda install pandas matplotlib`
- Review invalid Sem_GPA rows flagged in the cleaning cell.
- Adjust outlier capping / imputation strategy as needed.
- Add statistical tests or survival modeling using `Censored` and `Target_Dropout_Next_Sem`.
